# 4.3 每位置的一排數字，怎樣先整理尺度？


兩個位置的特徵是 `[1,2,3]` 與 `[10,20,30]`，相對排列相同，大小卻差十倍。送入後續配方前，常先讓每張卡各自整理尺度，避免數值任意放大。

**層正規化**叫 LayerNorm，簡稱 LN。每位置先減自己的平均，再除自己的標準差：第一排平均 2，減後 `[-1,0,1]`；平方偏離的平均是 2/3，標準差約 0.816，整理後約 `[-1.225,0,1.225]`。第二排用平均 20、標準差約 8.165，也得到近似相同結果。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

x = torch.tensor([[[1.0, 2.0, 3.0], [10.0, 20.0, 30.0]]])
layer = nn.LayerNorm(3)
y = layer(x)
print(y)
print("平均", y.mean(dim=-1))
print("變異數", y.var(dim=-1, unbiased=False))
assert torch.allclose(y.mean(dim=-1), torch.zeros(1, 2), atol=1e-6)

輸入形狀 `[1,2,3]` 表示一筆、兩位置、每位置三格。`LayerNorm(3)` 沿最後三格，給每張卡各算平均與變異數；不是沿兩個位置一起算。輸出兩排都接近上面的結果，平均約 0，變異數約 1。

實際分母是 `sqrt(變異數+epsilon)`，加小正數 epsilon 防止三格全相同時除以 0。之後每格還可乘自己的可調倍率 γ、加偏移 β；初始 γ=1、β=0，所以現在看到整理結果。若後來全設 γ=2、β=1，輸出平均會為 1、變異數接近 4。LN 是先整理再可調，最終不必永遠零平均。

這種計算不跨位置，也不依賴同批另一句。因此把第 1 位置每格加 100，第一位置不變；第二位置也因為平均同步加 100，整理結果近乎不變。用下面的小段核對：


In [ ]:
first = layer(x)
x[0, 1] += 100
shifted = layer(x)
print(torch.allclose(first[0, 0], shifted[0, 0]))
print(torch.allclose(first[0, 1], shifted[0, 1]))

兩行應都為 `True`。再從原始輸入重新開始，只把修改那行換成 `x[0,1,0]+=100`，只增第 1 位置的一格；第一行仍 True，第二行變 False。全排平移與格間關係改變，是不同操作。

如果誤跨位置算平均，前面位置可能間接使用後面資料，即使注意力遮罩正確也會偷看。用「每張位置卡自己整理」來核對軸，才能保住前文限制。

<details>
<summary>補充與重做</summary>

`dim=-1` 是最後特徵軸，`x[0,1]` 是第 0 筆、第 1 位置的整排，`x[0,1,0]` 才是其中一格。`unbiased=False` 用三格平方偏離的平均，與 LN 約定相同。變異數與標準差見 [3.5](https://birdhackor.github.io/tiny-perceptron-vlm/3.5.html)。公式為 `z_j=(x_j-m)/sqrt(v+epsilon)`、`y_j=γ_j*z_j+β_j`，j 是特徵格；同一格的 γ/β 在各位置共用。

</details>
